# Phase 1.5 — Cloud-Sorted Monthly HLS Composites

Adapts the workflow from `landsat_merge.ipynb` to our HLS pipeline:
search → cloud-sort → reproject onto a state grid → average overlapping
pixels → mask non-corn pixels → save GeoTIFF to S3.

```
search_hls_month → top-K cleanest scenes
       │
       ▼
reproject each onto EPSG:5070 / 30 m state grid
       │
       ▼
nanmean across scenes  (one denoised (6,H,W) mosaic)
       │
       ▼
mask non-corn pixels (CDL class 1)
       │
       ▼
s3://cornsight-data/processed/composites/{state}/{year}/{month}.tif
       │
       ▼
extract_features_from_array(model, composite)   ← 1 embedding / month
```

**Why this matters**: Phase 2 currently produces ~50 embeddings per
state-year (one per HLS granule, partially cloudy, fragmented). With
monthly composites, Prithvi sees one denoised, corn-only, full-state
input per month → 6 embeddings per state-year that line up directly
with the `month` axis used by `feature_fusion` and
`forecaster.DATE_MAP`.

In [ ]:
# Cell 1 — Setup
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import torch

from hls_downloader import login
from composite_builder import (
    build_monthly_composite,
    build_composites_for_year,
    load_composite_from_s3,
    composite_s3_key,
)

BUCKET = "cornsight-data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")

login()

## Cell 2 — Smoke test: one composite (Iowa, August 2023)

Should take ~1–3 minutes on CPU. Watches you exactly which scenes were
selected, what their cloud cover was, and the final corn-pixel coverage.

In [ ]:
# Cell 2 — Single composite
key = build_monthly_composite(
    state_abbr="IA",
    year=2023,
    month=8,
    bucket=BUCKET,
    top_k=5,
    mask_corn=True,
    overwrite=True,        # force rebuild while iterating
)
print("S3 key:", key)

## Cell 3 — Visualize the composite

Plots a false-colour preview (NIR / Red / Green) plus the per-band
coverage so you can see how much of the state ended up with valid corn
pixels.

In [ ]:
# Cell 3 — Visualize
import matplotlib.pyplot as plt

arr, profile = load_composite_from_s3(BUCKET, "IA", 2023, 8)
print(f"shape = {arr.shape}   crs = {profile['crs']}   res = {profile['transform'].a:.1f} m")

# Bands: B02 Blue, B03 Green, B04 Red, B05 NIR, B06 SWIR1, B07 SWIR2
def stretch(b):
    finite = b[np.isfinite(b)]
    if finite.size == 0:
        return np.zeros_like(b)
    lo, hi = np.percentile(finite, [2, 98])
    return np.clip((b - lo) / max(hi - lo, 1e-6), 0, 1)

rgb = np.dstack([stretch(arr[3]), stretch(arr[2]), stretch(arr[1])])  # NIR/Red/Green

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(rgb)
axes[0].set_title("IA 2023-08 composite (NIR / Red / Green)")
axes[0].axis("off")

valid_pct = 100.0 * np.isfinite(arr).mean(axis=(1, 2))
axes[1].bar(["B02", "B03", "B04", "B05", "B06", "B07"], valid_pct)
axes[1].set_ylabel("% pixels with valid corn data")
axes[1].set_ylim(0, 100)
axes[1].set_title("Per-band corn coverage")
plt.tight_layout(); plt.show()

## Cell 4 — Build all 6 monthly composites for one state-year

Once cell 2 looks reasonable, fill in the full growing season for IA 2023.
`build_composites_for_year` skips months already present in S3, so it's
safe to rerun if interrupted.

In [ ]:
# Cell 4 — Full season for one state
build_composites_for_year(
    state_abbr="IA",
    year=2023,
    bucket=BUCKET,
    months=(5, 6, 7, 8, 9, 10),
    top_k=5,
    mask_corn=True,
    overwrite=False,
)

## Cell 5 — Run Prithvi against the composites

Same `extract_features_from_array` as before; the input is now a
denoised, corn-only mosaic instead of a raw granule. Embeddings are
saved with a synthetic granule id of `composite_{month:02d}` so the
existing `feature_fusion` / `month`-grouping code keeps working
unchanged.

In [ ]:
# Cell 5 — Composite → Prithvi embedding
from privthi_extractor import (
    load_prithvi,
    extract_features_from_array,
    save_features_to_s3,
)

model = load_prithvi(device=DEVICE)

STATE  = "IA"
YEAR   = 2023
MONTHS = (5, 6, 7, 8, 9, 10)
FORECAST_DATE = "final"   # writes to processed/features/{state}/{year}/final/

for month in MONTHS:
    try:
        arr, profile = load_composite_from_s3(BUCKET, STATE, YEAR, month)
    except Exception as e:
        print(f"  {month:02d}: composite missing — {e}")
        continue

    # Replace NaN with 0 *after* normalization happens inside Prithvi.
    # extract_features_from_array does its own normalization, so we just
    # make sure no NaNs reach the conv layers.
    arr_clean = np.nan_to_num(arr, nan=0.0).astype(np.float32)

    feats = extract_features_from_array(model, arr_clean, max_tiles=64)
    gid = f"composite_{month:02d}"
    save_features_to_s3(feats, gid, BUCKET, STATE, YEAR, FORECAST_DATE)
    print(f"  {month:02d}: features shape={feats.shape}  mean={feats.mean():+.3f}  -> {gid}")

## Cell 6 — Full sweep: 5 states × N years × 6 months

This is the production loop. Each (state, year) is a few minutes of
network I/O + reprojection + a handful of Prithvi forward passes, so
budget ~5–10 min per (state, year) on CPU and a fraction of that on
GPU.

In [ ]:
# Cell 6 — Full sweep
STATES = ["IA", "CO", "WI", "MO", "NE"]
YEARS  = range(2015, 2025)
FORECAST_DATE = "final"

for state in STATES:
    for year in YEARS:
        print(f"\n=== {state} {year} ===")
        build_composites_for_year(
            state_abbr=state, year=year, bucket=BUCKET,
            months=(5, 6, 7, 8, 9, 10),
            top_k=5, mask_corn=True, overwrite=False,
        )
        for month in (5, 6, 7, 8, 9, 10):
            try:
                arr, _ = load_composite_from_s3(BUCKET, state, year, month)
            except Exception:
                continue
            arr_clean = np.nan_to_num(arr, nan=0.0).astype(np.float32)
            feats = extract_features_from_array(model, arr_clean, max_tiles=64)
            save_features_to_s3(
                feats, f"composite_{month:02d}",
                BUCKET, state, year, FORECAST_DATE,
            )
            print(f"  {state} {year}-{month:02d}: saved feature {feats.shape}")

## Where this slots into the rest of the pipeline

- `feature_fusion.fuse_features` already keys on `(year, month, county, fips,
  state)` — the only change is that `prithvi_embedding` now has one row per
  (state, year, month) instead of one per granule. Counties within a state
  share the same composite-derived embedding for that month, which is fine
  because Prithvi has no sub-state spatial localisation in our pooled
  embedding anyway.
- `forecaster.get_analog_years` is unchanged — its `DATE_MAP` already
  filters by `month`.
- `train_forecast.ipynb` is unchanged — it reads `master_features.parquet`.

## Interactive Folium map

Renders the (state, year, month) composite as a true-color RGB tile on top of an OpenStreetMap basemap. Non-corn pixels are transparent. Pan/zoom to inspect — the embedding the model sees is exactly what's drawn on this layer.

In [ ]:
import io, base64
import numpy as np
import folium
from PIL import Image
from rasterio.warp import transform_bounds
from composite_builder import load_composite_from_s3

STATE = "IA"; YEAR = 2023; MONTH = 8

arr, profile = load_composite_from_s3("cornsight-data", STATE, YEAR, MONTH)
print(f"loaded composite: shape={arr.shape}  finite={100*np.isfinite(arr[0]).mean():.1f}%")

# Build true-color RGB (HLS bands: 0=B,1=G,2=R,3=NIR,4=SWIR1,5=SWIR2)
def stretch(b, lo=2, hi=98):
    finite = b[np.isfinite(b)]
    if finite.size == 0:
        return np.zeros_like(b, dtype=np.uint8)
    p_lo, p_hi = np.percentile(finite, [lo, hi])
    out = np.clip((b - p_lo) / max(p_hi - p_lo, 1e-6), 0, 1)
    out = np.where(np.isfinite(b), out, 0)
    return (out * 255).astype(np.uint8)

R = stretch(arr[2]); G = stretch(arr[1]); B = stretch(arr[0])
alpha = (np.isfinite(arr[0]).astype(np.uint8)) * 255  # transparent where non-corn / no data
rgba = np.dstack([R, G, B, alpha])

# Encode as PNG -> data URL
buf = io.BytesIO()
Image.fromarray(rgba, "RGBA").save(buf, format="PNG", optimize=True)
png_b64 = base64.b64encode(buf.getvalue()).decode()
data_url = f"data:image/png;base64,{png_b64}"
print(f"PNG size: {len(buf.getvalue())/1e6:.2f} MB")

# Reproject bounds from EPSG:5070 -> EPSG:4326 for folium
left, bottom, right, top = profile["transform"] * (0, arr.shape[1]) + profile["transform"] * (arr.shape[2], 0)
# transform.__mul__ returns (x,y); rebuild bbox properly:
t = profile["transform"]
W, H = arr.shape[2], arr.shape[1]
xs = [t * (0,0), t * (W,0), t * (W,H), t * (0,H)]
left   = min(p[0] for p in xs); right = max(p[0] for p in xs)
bottom = min(p[1] for p in xs); top   = max(p[1] for p in xs)
ll = transform_bounds(profile["crs"], "EPSG:4326", left, bottom, right, top)
w, s, e, n = ll
print(f"bounds (lon/lat): W={w:.3f} S={s:.3f} E={e:.3f} N={n:.3f}")

m = folium.Map(location=[(s+n)/2, (w+e)/2], zoom_start=7, tiles="OpenStreetMap")
folium.raster_layers.ImageOverlay(
    image=data_url,
    bounds=[[s, w], [n, e]],
    opacity=0.85,
    name=f"{STATE} {YEAR}-{MONTH:02d} corn composite (true color)",
).add_to(m)
folium.LayerControl().add_to(m)
m.save("../corn_composite_map.html")
print("saved -> corn_composite_map.html")
m
